# Predictive Maintenance Sensor Anomaly Detection

Detect abnormal equipment states from multivariate sensor readings.

**Portfolio category:** Anomaly detection

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Demonstration data with labels hidden from fitting

In [ ]:
n_normal, n_anomaly = 2850, 150
normal = pd.DataFrame({
    "vibration": rng.normal(2.0, 0.25, n_normal),
    "temperature": rng.normal(68, 4, n_normal),
    "pressure": rng.normal(31, 1.8, n_normal),
    "rotation_speed": rng.normal(1450, 65, n_normal),
    "current": rng.normal(11.5, 0.8, n_normal),
})
anomalies = pd.DataFrame({
    "vibration": rng.normal(4.8, 0.8, n_anomaly),
    "temperature": rng.normal(91, 8, n_anomaly),
    "pressure": rng.normal(24, 4, n_anomaly),
    "rotation_speed": rng.normal(1180, 180, n_anomaly),
    "current": rng.normal(16.5, 2.1, n_anomaly),
})

normal["hidden_label"] = 0
anomalies["hidden_label"] = 1
data = pd.concat([normal, anomalies], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
feature_names = [column for column in data.columns if column != "hidden_label"]
display(data.head())

## 3. Data quality and baseline prevalence

In [ ]:
display(data[feature_names].describe().T)
print("Missing cells:", int(data[feature_names].isna().sum().sum()))
hidden_prevalence = data["hidden_label"].mean()
print(f"Hidden evaluation prevalence: {hidden_prevalence:.3%}")

## 4. Feature distributions

In [ ]:
data[feature_names].hist(figsize=(12, 7), bins=30)
plt.suptitle("Feature distributions", y=1.02)
plt.tight_layout()

## 5. Fit unsupervised detectors

In [ ]:
X = StandardScaler().fit_transform(data[feature_names])
contamination = max(0.01, min(0.15, hidden_prevalence))
isolation = IsolationForest(
    n_estimators=300,
    contamination=contamination,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
isolation.fit(X)
isolation_score = -isolation.decision_function(X)

lof = LocalOutlierFactor(n_neighbors=35, contamination=contamination)
lof.fit_predict(X)
lof_score = -lof.negative_outlier_factor_

## 6. Evaluate rankings after fitting

In [ ]:
y = data["hidden_label"].to_numpy()
k = max(1, int(y.sum()))
rows = []
for name, scores in {"Isolation Forest": isolation_score, "Local Outlier Factor": lof_score}.items():
    top_k = np.argsort(scores)[-k:]
    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y, scores),
        "average_precision": average_precision_score(y, scores),
        "precision_at_k": y[top_k].mean(),
        "alert_rate": k / len(y),
    })
evaluation = pd.DataFrame(rows).sort_values("average_precision", ascending=False)
display(evaluation.round(4))
best_name = evaluation.iloc[0]["model"]
best_score = isolation_score if best_name == "Isolation Forest" else lof_score
data["anomaly_score"] = best_score
data["flagged"] = False
data.loc[np.argsort(best_score)[-k:], "flagged"] = True

## 7. Score diagnostics

In [ ]:
sns.histplot(data=data, x="anomaly_score", hue="hidden_label", bins=45, element="step", stat="density", common_norm=False)
plt.title("Anomaly score distribution; labels shown only for evaluation")
plt.tight_layout()

## 8. Inspect the highest-risk observations

In [ ]:
display(data.sort_values("anomaly_score", ascending=False).head(15).round(3))

## 9. Key findings

Precision at k is often more operationally useful than a default anomaly threshold because review teams have finite capacity.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For predictive maintenance sensor anomaly detection,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.